# Verdict recomputation from saved metrics

Every trained run wrote a per-patch metrics JSON. This notebook reads those
files and nothing else. It trains nothing, samples nothing and touches no
GPU, so it is **fully deterministic** -- running it twice gives identical
output, which is not true of any other notebook in this project.

## Why this exists

The comparison cells inside `14`, `15` and `16` each print a verdict computed
from whatever was in memory at the time, against a threshold that changed as
the project went on:

* `14` used a floor of 0.0301, taken from a single pairwise gap, and compared
  against `09` alone rather than against the three-run mean.
* `15` and `16` compared against the three-run mean correctly, but divided by
  the standard deviation of a *single run* (0.0262) instead of the standard
  deviation of a *difference against a three-run mean* (0.0303), and took no
  account of having only two degrees of freedom.

None of that affects a single measured number. The metrics are what they are.
Only the verdict lines are wrong, and they are recomputed here on one
consistent basis.

## Why the source notebooks cannot simply be re-run

Their comparison cells depend on `metric_rows`, which exists only after the
evaluation cell has run. Re-running the evaluation cell re-samples the
diffusion model from fresh Gaussian noise, so it returns *different* metrics
from the same checkpoint. Re-running training changes them further still --
that variation is precisely what the seed study measured.

Re-running is therefore not a way to reproduce these numbers. Reading the
saved files is.

## Setup

In [ ]:
import json
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path('/cs/student/project_msc/2025/aibh/jiayiche/s1_training_outputs')

# The three runs of the k=3 real-attribute configuration. Identical split,
# identical hyperparameters, initialisation seed varied. The third is the
# inert-DEM run, which is a seed replicate because its extra input branch
# was measured to carry no spatial information (dem_unet/01).
SEED_RUNS = {
    '09, seed 42': 's1_pcrtc_realattrs_spatialsplit_validation_metrics.json',
    'seed 43': 's1_pcrtc_realattrs_spatialsplit_seed43_validation_metrics.json',
    'inert-DEM branch': 's1_pcrtc_dem_realattrs_spatialsplit_validation_metrics.json',
}

# Everything else, compared against that distribution.
RUNS = {
    'baseline (repeat, zeros)':   's1_pcrtc_baseline_spatialsplit_validation_metrics.json',
    'native 2ch, zeros':          's1_pcrtc_native2ch_spatialsplit_validation_metrics.json',
    'native 2ch, real attrs':     's1_pcrtc_native2ch_realattrs_spatialsplit_validation_metrics.json',
    'k = 7 views':                's1_pcrtc_realattrs_spatialsplit_k7_validation_metrics.json',
    'unseen dates':               's1_pcrtc_realattrs_unseen_date_metrics.json',
    'Cambridge Bay':              's1_pcrtc_cambridgebay_crossregion_metrics.json',
    'EW swath':                   's1_ew_realattrs_spatialsplit_validation_metrics.json',
    'leaky split (06)':           's1_pcrtc_realattrs_validation_metrics.json',
}

METRICS = ['zncc', 'rmse_m', 'bias_m', 'sigma_error_pct',
           'normal_angle_error_deg', 'jsd', 'psd_rmse',
           'gt_std_val', 'pred_std_val']


def load(name):
    p = OUTPUT_DIR / name
    if not p.exists():
        return None
    return json.load(p.open())


def means(rows):
    return {m: float(np.nanmean([r[m] for r in rows if m in r])) for m in METRICS}

## The reference distribution

Mean, standard deviation and the two thresholds. Nothing below chooses a
threshold after seeing a result: both are fixed here and applied uniformly.

In [ ]:
seed_means, missing = {}, []
for label, fname in SEED_RUNS.items():
    rows = load(fname)
    if rows is None:
        missing.append(fname)
        continue
    seed_means[label] = means(rows)['zncc']

if missing:
    print('MISSING seed-run files -- the thresholds below cannot be trusted:')
    for m in missing:
        print('   ', m)

vals = np.array(list(seed_means.values()))
assert len(vals) >= 3, 'Need all three seed runs to estimate the spread.'

MEAN = vals.mean()
SD = vals.std(ddof=1)
N = len(vals)

# A single new run compared against the mean of N reference runs:
#   Var(x - xbar) = sd^2 + sd^2/N
SD_DIFF = SD * np.sqrt(1.0 + 1.0 / N)

# sd is estimated from N observations, so N-1 degrees of freedom.
T_CRIT = 4.303          # t(0.975, df=2)
STRICT = T_CRIT * SD_DIFF
LOOSE = 2.0 * SD_DIFF

print('k=3 real-attribute configuration, three runs:')
for label, v in seed_means.items():
    print(f'  {label:<22} {v:.4f}')
print()
print(f'  mean                   {MEAN:.4f}')
print(f'  s.d. (single run)      {SD:.4f}')
print(f'  range                  {vals.max() - vals.min():.4f}')
print()
print(f'  s.d. of a difference vs the mean of {N}:  {SD_DIFF:.4f}')
print(f'  threshold, 2 s.d.  (permissive)        : {LOOSE:.4f}')
print(f'  threshold, t(0.975, {N - 1}) (95% honest)    : {STRICT:.4f}')
print()
print('  The strict threshold accounts for s.d. being estimated from three')
print('  observations. It is the one quoted in the write-up.')

## Every run against that distribution

`delta` is the difference in mean ZNCC from the three-run mean. `s.d.` is
that difference expressed in units of `SD_DIFF`.

In [ ]:
print(f'{"run":<28}{"ZNCC":>9}{"delta":>10}{"s.d.":>8}   verdict')
print('-' * 78)

table_rows = []
for label, fname in RUNS.items():
    rows = load(fname)
    if rows is None:
        print(f'{label:<28}{"--":>9}   file not found: {fname}')
        continue
    m = means(rows)
    z = m['zncc']
    d = z - MEAN
    n_sd = d / SD_DIFF
    if abs(d) >= STRICT:
        verdict = 'RESOLVABLE'
    elif abs(d) >= LOOSE:
        verdict = 'marginal (permissive only)'
    else:
        verdict = 'not resolvable'
    print(f'{label:<28}{z:>9.4f}{d:>+10.4f}{n_sd:>8.2f}   {verdict}')
    table_rows.append((label, m, d, n_sd, verdict))

print()
print('"not resolvable" means the difference is smaller than initialisation')
print('seed alone reliably produces. It is not a claim that the effect is zero.')

## Full metric table

The numbers Chapter 4 reports, in one place, including the three seed runs.

In [ ]:
hdr = f'{"run":<28}' + ''.join(f'{m.replace("_", " "):>13}' for m in METRICS)
print(hdr)
print('-' * len(hdr))
for label, fname in list(SEED_RUNS.items()) + list(RUNS.items()):
    rows = load(fname)
    if rows is None:
        continue
    m = means(rows)
    print(f'{label:<28}' + ''.join(f'{m[k]:>13.4f}' for k in METRICS))

## LaTeX table body

Paste into `tables.tex`. Column order matches the header printed above.

In [ ]:
NICE = {'zncc': 'ZNCC', 'rmse_m': 'RMSE (m)', 'sigma_error_pct': '$\\sigma$ err (\\%)',
        'jsd': 'JSD', 'psd_rmse': 'PSD', 'pred_std_val': 'pred std (m)'}
COLS = ['zncc', 'rmse_m', 'sigma_error_pct', 'jsd', 'psd_rmse', 'pred_std_val']

print('\\begin{tabular}{l' + 'r' * len(COLS) + '}')
print('\\toprule')
print('Run & ' + ' & '.join(NICE[c] for c in COLS) + ' \\\\')
print('\\midrule')
for label, fname in list(SEED_RUNS.items()) + list(RUNS.items()):
    rows = load(fname)
    if rows is None:
        continue
    m = means(rows)
    safe = label.replace('&', '\\&')
    print(f'{safe} & ' + ' & '.join(f'{m[c]:.4f}' for c in COLS) + ' \\\\')
print('\\bottomrule')
print('\\end{tabular}')